# Analítica de Datos para la Estimación y Gestión del Riesgo en Costos Médicos
**Salud Vital S.A. · SCY1102 · Caso 2** · Ciclo completo: **Datos → Modelo → Evaluación → Operación**

Este notebook genera paso a paso la aplicación: prepara los datos, entrena y evalúa el modelo, segmenta el riesgo, simula el monitoreo con semáforo, arma el tablero y despliega una app web (Gradio) para estimar el costo médico anual esperado.

> El modelo es una **herramienta de apoyo actuarial** (human-in-the-loop). No reemplaza el criterio del comité técnico.

## 0. Configuración

In [ ]:
!pip -q install gradio

In [ ]:
import warnings, json, hashlib, os, io, sys, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn, joblib
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

def clp(x):
    return "$" + f"{x:,.0f}".replace(",", ".")

print("Entorno listo · sklearn", sklearn.__version__)

Entorno listo · sklearn 1.6.1


## 1. DATOS · Carga, validación y documentación
Variables: `age`, `sex`, `bmi`, `children`, `smoker`, `region` y la variable objetivo `charges` (costo médico anual).
Si estás en Colab te pedirá subir `insurance.csv`; si cancelas, lo descarga desde un repositorio público (el mismo dataset del libro de Lantz citado en el informe).

In [ ]:
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

def cargar_datos():
    try:
        from google.colab import files
        print("Sube insurance.csv (o cancela para descargarlo automáticamente)")
        subido = files.upload()
        if subido:
            return pd.read_csv(io.BytesIO(next(iter(subido.values()))))
    except ImportError:
        pass
    if os.path.exists("insurance.csv"):
        return pd.read_csv("insurance.csv")
    return pd.read_csv(URL)

df = cargar_datos()

# El informe trabaja en CLP: el dataset original viene en USD y se escala x1000.
# Si el archivo ya está en CLP (valores de millones) no se toca.
ESCALA_CLP = 1000
if df["charges"].mean() < 100_000:
    df["charges"] = df["charges"] * ESCALA_CLP
    print(f"charges escalado x{ESCALA_CLP} para quedar en CLP, igual que el informe")

print("Registros:", len(df), "| columnas:", list(df.columns))
print("Costo medio fumadores:", clp(df.loc[df.smoker == "yes", "charges"].mean()),
      "(informe: $32.050.232)")
print("Costo medio no fumadores:", clp(df.loc[df.smoker == "no", "charges"].mean()),
      "(informe: $8.434.268)")
df.head()

Sube insurance.csv (o cancela para descargarlo automáticamente)


KeyboardInterrupt: 

In [ ]:
# --- Diagnóstico de calidad (se documenta cada decisión) ---
calidad = pd.DataFrame({"tipo": df.dtypes.astype(str),
                        "nulos": df.isna().sum(),
                        "valores_unicos": df.nunique()})
display(calidad)

reglas = {
    "edad entre 18 y 64":     df.age.between(18, 64).all(),
    "IMC entre 10 y 60":      df.bmi.between(10, 60).all(),
    "hijos >= 0":             (df.children >= 0).all(),
    "costo > 0":              (df.charges > 0).all(),
    "sex en {male, female}":  df.sex.isin(["male", "female"]).all(),
    "smoker en {yes, no}":    df.smoker.isin(["yes", "no"]).all(),
    "4 regiones esperadas":   df.region.nunique() == 4,
}
print(pd.Series(reglas).map({True: "OK", False: "REVISAR"}))

dups = df.duplicated().sum()
print("\nDuplicados exactos:", dups, "(se conservan: pueden ser afiliados distintos con el mismo perfil)")

registro_decisiones = {
    "nulos": int(df.isna().sum().sum()),
    "duplicados": int(dups),
    "outliers_costo": "se conservan: son siniestros reales de alto costo, no errores",
}
registro_decisiones

NameError: name 'df' is not defined

### Análisis exploratorio

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(df.charges / 1e6, bins=40, kde=True, ax=ax[0, 0])
ax[0, 0].set(title="Distribución del costo anual", xlabel="Millones CLP")

sns.boxplot(data=df, x="smoker", y=df.charges / 1e6, ax=ax[0, 1])
ax[0, 1].set(title="Costo según tabaquismo", ylabel="Millones CLP")

sns.scatterplot(data=df, x="bmi", y=df.charges / 1e6, hue="smoker", alpha=.6, ax=ax[1, 0])
ax[1, 0].set(title="IMC vs costo (interacción fumar x obesidad)", ylabel="Millones CLP")

sns.barplot(data=df, x="region", y=df.charges / 1e6, estimator="mean", errorbar=None, ax=ax[1, 1])
ax[1, 1].set(title="Costo medio por región", ylabel="Millones CLP")
plt.tight_layout(); plt.show()

df["obeso"] = df.bmi >= 30
print(df.groupby(["smoker", "obeso"]).charges.agg(["count", "mean"]).round(0))

## 2. MODELO · Variable objetivo, fuga de datos, partición y trazabilidad
- **Objetivo:** `charges`. **Riesgo de fuga:** ninguna variable se calcula a partir del costo; la variable derivada `fumador_obeso` usa solo `smoker` e `bmi`.
- **Partición:** 80 % entrenamiento / 20 % prueba, estratificada por tabaquismo y con semilla fija.
- **Trazabilidad:** se guarda hash del dataset, semilla y versiones de librerías.

In [ ]:
df["fumador_obeso"] = ((df.smoker == "yes") & (df.bmi >= 30)).astype(int)

NUM = ["age", "bmi", "children"]
CAT = ["sex", "smoker", "region"]
FEATURES = NUM + CAT + ["fumador_obeso"]
TARGET = "charges"

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=df["smoker"])
print("Train:", X_train.shape, "| Test:", X_test.shape)

trazabilidad = {
    "hash_dataset_md5": hashlib.md5(pd.util.hash_pandas_object(df[FEATURES + [TARGET]]).values.tobytes()).hexdigest(),
    "semilla": SEED, "n_train": len(X_train), "n_test": len(X_test),
    "python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__,
}
trazabilidad

In [ ]:
pre = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(drop="first"), CAT),
], remainder="passthrough")

def pipe(modelo):
    return Pipeline([("pre", pre), ("modelo", modelo)])

candidatos = {
    "Regresión lineal": pipe(LinearRegression()),
    "Ridge":            pipe(Ridge(alpha=1.0)),
    "Random Forest":    pipe(RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                   random_state=SEED, n_jobs=-1)),
    "Gradient Boosting": pipe(GradientBoostingRegressor(random_state=SEED)),
}

# Búsqueda de hiperparámetros del candidato principal
grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [2, 3],
    "modelo__learning_rate": [0.03, 0.1],
    "modelo__min_samples_leaf": [10, 20],
}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
gs = GridSearchCV(pipe(GradientBoostingRegressor(random_state=SEED)), grid, cv=cv,
                  scoring="neg_root_mean_squared_error", n_jobs=-1)
gs.fit(X_train, y_train)
print("Mejores hiperparámetros GB:", gs.best_params_)
candidatos["Gradient Boosting (ajustado)"] = gs.best_estimator_

filas = []
for nombre, m in candidatos.items():
    r = cross_validate(m, X_train, y_train, cv=cv,
                       scoring=("neg_root_mean_squared_error", "r2"))
    filas.append({"modelo": nombre,
                  "RMSE_cv": -r["test_neg_root_mean_squared_error"].mean(),
                  "R2_cv": r["test_r2"].mean(),
                  "R2_sd": r["test_r2"].std()})
comparacion = pd.DataFrame(filas).sort_values("RMSE_cv").reset_index(drop=True)
comparacion["RMSE_cv"] = comparacion["RMSE_cv"].round(0)
display(comparacion.round(3))

mejor = comparacion.loc[0, "modelo"]
modelo = candidatos[mejor].fit(X_train, y_train)
print("Modelo seleccionado:", mejor)

## 3. EVALUACIÓN · Métricas, sobreajuste, equidad e importancia
Métrica principal: **MAE** (error promedio en pesos, fácil de explicar a Actuaría) junto con **R²**. El sobreajuste se detecta comparando entrenamiento vs prueba.

In [ ]:
pred_tr, pred_te = modelo.predict(X_train), modelo.predict(X_test)
res = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Prueba"],
    "MAE":  [mean_absolute_error(y_train, pred_tr), mean_absolute_error(y_test, pred_te)],
    "RMSE": [mean_squared_error(y_train, pred_tr) ** .5, mean_squared_error(y_test, pred_te) ** .5],
    "R2":   [r2_score(y_train, pred_tr), r2_score(y_test, pred_te)],
})
display(res.round(3))
brecha = res.R2[0] - res.R2[1]
print(f"Brecha R2 train-test: {brecha:.3f} ->",
      "posible sobreajuste" if brecha > 0.10 else "sin señales fuertes de sobreajuste")

R2_BASE = r2_score(y_test, pred_te)
MAE_BASE = mean_absolute_error(y_test, pred_te)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(y_test / 1e6, pred_te / 1e6, alpha=.5)
lim = [0, max(y_test.max(), pred_te.max()) / 1e6]
ax[0].plot(lim, lim, "r--"); ax[0].set(title="Real vs predicho (prueba)", xlabel="Real (M CLP)", ylabel="Predicho (M CLP)")
sns.histplot((y_test - pred_te) / 1e6, bins=30, kde=True, ax=ax[1])
ax[1].set(title="Residuos", xlabel="Real - predicho (M CLP)")
plt.tight_layout(); plt.show()

In [ ]:
# --- Equidad: error por subgrupo (el modelo no debe fallar sistemáticamente en un grupo) ---
t = X_test.copy()
t["real"], t["pred"] = y_test, pred_te
t["tramo_edad"] = pd.cut(t.age, [17, 30, 45, 65], labels=["18-30", "31-45", "46-65"])

def por_grupo(col):
    g = t.groupby(col).apply(lambda d: pd.Series({
        "n": len(d),
        "MAE": mean_absolute_error(d.real, d.pred),
        "sesgo_%": (d.pred.mean() - d.real.mean()) / d.real.mean() * 100}))
    g.index = [f"{col}={i}" for i in g.index]
    return g

equidad = pd.concat([por_grupo(c) for c in ["smoker", "sex", "region", "tramo_edad"]])
equidad["alerta"] = np.where(equidad["sesgo_%"].abs() > 10, "REVISAR", "ok")
display(equidad.round(1))

imp = permutation_importance(modelo, X_test, y_test, n_repeats=10, random_state=SEED, scoring="r2")
imp = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(title="Importancia de variables (permutación, R2)"); plt.show()

## 4. SEGMENTACIÓN DE RIESGO
Se clasifica a cada afiliado según su costo esperado. Los umbrales se fijan con el entrenamiento (percentiles 50 y 85) y se reutilizan en producción.

In [ ]:
UMB_MEDIO, UMB_ALTO = np.quantile(modelo.predict(X_train), [0.50, 0.85])

def segmento(costo):
    return "Alto" if costo >= UMB_ALTO else ("Medio" if costo >= UMB_MEDIO else "Bajo")

seg = df.copy()
seg["costo_esperado"] = modelo.predict(df[FEATURES])
seg["segmento"] = seg.costo_esperado.map(segmento)

perfil = seg.groupby("segmento").agg(
    afiliados=("age", "size"), edad_media=("age", "mean"), imc_medio=("bmi", "mean"),
    pct_fumadores=("smoker", lambda s: (s == "yes").mean() * 100),
    costo_esperado=("costo_esperado", "mean"), costo_real=("charges", "mean"),
).reindex(["Bajo", "Medio", "Alto"]).round(1)
display(perfil)

## 5. OPERACIÓN · Monitoreo con semáforo (dos señales)
Se simula un año de producción. Se mira **(a)** cambio en los datos de entrada (PSI) y **(b)** desempeño del modelo (R² mensual), porque los datos suelen cambiar *antes* de que el desempeño caiga.
- Semáforo desempeño (R²): verde ≥ 0,70 · amarillo 0,60–0,70 · rojo < 0,60
- Semáforo datos (PSI máx.): verde < 0,10 · amarillo 0,10–0,25 · rojo ≥ 0,25

*La simulación es ilustrativa: desde el mes 5 llegan más afiliados con IMC alto y fumadores (cambio de datos) y desde el mes 8 suben los costos reales de los prestadores (cambio de concepto).*

In [ ]:
def psi(ref, new, bins=5):
    ref, new = np.asarray(ref, float), np.asarray(new, float)
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, edges)[0] / len(ref) + 1e-4
    q = np.histogram(new, edges)[0] / len(new) + 1e-4
    return float(np.sum((q - p) * np.log(q / p)))

def psi_binaria(ref, new):
    p = np.array([1 - ref.mean(), ref.mean()]) + 1e-4
    q = np.array([1 - new.mean(), new.mean()]) + 1e-4
    return float(np.sum((q - p) * np.log(q / p)))

U_R2 = {"amarillo": 0.70, "rojo": 0.60}
U_PSI = {"amarillo": 0.10, "rojo": 0.25}

def color_r2(v):  return "ROJO" if v < U_R2["rojo"] else ("AMARILLO" if v < U_R2["amarillo"] else "VERDE")
def color_psi(v): return "ROJO" if v >= U_PSI["rojo"] else ("AMARILLO" if v >= U_PSI["amarillo"] else "VERDE")
ORDEN = {"VERDE": 0, "AMARILLO": 1, "ROJO": 2}

rng = np.random.default_rng(SEED)
pool = X_test.copy(); pool["real"] = y_test.values
N_MES = 300
registros = []
for mes in range(1, 13):
    w = np.ones(len(pool))
    if mes >= 5:                                   # cambio en los datos de entrada
        k = 0.5 * (mes - 4)
        w = 1 + k * (pool.bmi >= 30).values + k * (pool.smoker == "yes").values
    muestra = pool.sample(N_MES, replace=True, weights=w, random_state=int(rng.integers(1e9))).copy()
    if mes >= 8:                                   # cambio de concepto: suben los costos reales
        muestra["real"] = muestra["real"] * (1 + 0.15 * (mes - 7))
    p = modelo.predict(muestra[FEATURES])
    r2m = r2_score(muestra.real, p)
    psi_bmi = psi(X_train.bmi, muestra.bmi)
    psi_age = psi(X_train.age, muestra.age)
    psi_smk = psi_binaria((X_train.smoker == "yes").astype(int), (muestra.smoker == "yes").astype(int))
    psi_max = max(psi_bmi, psi_age, psi_smk)
    c_r2, c_psi = color_r2(r2m), color_psi(psi_max)
    estado = max([c_r2, c_psi], key=ORDEN.get)
    registros.append({"mes": mes, "R2": r2m, "MAE": mean_absolute_error(muestra.real, p),
                      "PSI_max": psi_max, "semaforo_datos": c_psi,
                      "semaforo_desempeno": c_r2, "estado": estado})
monitoreo = pd.DataFrame(registros)
display(monitoreo.round(3))

primer_cambio = monitoreo.loc[monitoreo.semaforo_datos != "VERDE", "mes"].min()
primer_caida = monitoreo.loc[monitoreo.semaforo_desempeno == "ROJO", "mes"].min()
print("Primer aviso por cambio en los datos:", "mes %d" % primer_cambio if pd.notna(primer_cambio) else "ninguno")
print("Primera caída roja del desempeño:", "mes %d" % primer_caida if pd.notna(primer_caida) else "aún no ocurre")

In [ ]:
# Matriz de control: cuando un indicador sale de su umbral, el plan reacciona
matriz_control = pd.DataFrame([
    ["R2 mensual", "< 0,70 (amarillo) / < 0,60 (rojo)",
     "Amarillo: se registra riesgo R-02. Rojo: se agrega la tarea 'reentrenar el modelo'",
     "Jefe de proyecto, reunión semanal de seguimiento"],
    ["PSI máximo de variables", ">= 0,10 (amarillo) / >= 0,25 (rojo)",
     "Se revisa la calidad de la ingesta (R-01) y se evalúa reentrenar con datos nuevos",
     "Ingeniero de datos / MLOps, revisión semanal"],
    ["Sesgo por subgrupo", "|sesgo| > 10 %",
     "Se registra R-03, se ajusta el modelo y se reporta al comité actuarial",
     "Consultor actuarial, auditoría semestral"],
], columns=["Indicador técnico", "Umbral que lo dispara", "Qué se registra o cambia en el plan", "Quién decide y cuándo"])
display(matriz_control)

### Dashboards: tablero técnico y tablero ejecutivo

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(monitoreo.mes, monitoreo.R2, marker="o", label="R2 mensual")
ax[0].axhline(U_R2["amarillo"], color="orange", ls="--", label="Amarillo 0,70")
ax[0].axhline(U_R2["rojo"], color="red", ls="--", label="Rojo 0,60")
ax[0].set(title="Tablero técnico · Desempeño", xlabel="Mes", ylabel="R2"); ax[0].legend()

ax[1].bar(monitoreo.mes, monitoreo.PSI_max, color="steelblue")
ax[1].axhline(U_PSI["amarillo"], color="orange", ls="--"); ax[1].axhline(U_PSI["rojo"], color="red", ls="--")
ax[1].set(title="Tablero técnico · Cambio en los datos (PSI)", xlabel="Mes", ylabel="PSI máximo")
plt.tight_layout(); plt.show()

ultimo = monitoreo.iloc[-1]
alto = (seg.segmento == "Alto").mean() * 100
print("TABLERO EJECUTIVO")
print(f"  Afiliados en riesgo alto: {alto:.0f}%  |  costo esperado promedio en ese grupo: {clp(perfil.loc['Alto', 'costo_esperado'])}")
print(f"  Error promedio del modelo (MAE): {clp(MAE_BASE)} por afiliado al año")
print(f"  Estado del semáforo (mes {int(ultimo.mes)}): {ultimo.estado}")
print("  Mensaje para la gerencia: el modelo ordena a los afiliados por riesgo de gasto y apoya la decisión del equipo actuarial; "
      "cuando el semáforo se pone rojo, hay que reentrenarlo antes de usarlo para fijar primas.")

## 6. DESPLIEGUE · Guardar el modelo y lanzar la aplicación
Se guarda el modelo con sus metadatos y se publica una app web (Gradio) con un enlace público temporal.

In [ ]:
os.makedirs("artefactos", exist_ok=True)
joblib.dump(modelo, "artefactos/modelo_costos.joblib")
metadatos = {"modelo": mejor, "features": FEATURES, "umbral_medio": float(UMB_MEDIO), "umbral_alto": float(UMB_ALTO),
             "R2_prueba": float(R2_BASE), "MAE_prueba": float(MAE_BASE), "trazabilidad": trazabilidad}
with open("artefactos/metadatos.json", "w") as f:
    json.dump(metadatos, f, indent=2)
print("Guardado en /artefactos:", os.listdir("artefactos"))

In [ ]:
import gradio as gr

modelo_app = joblib.load("artefactos/modelo_costos.joblib")

def estimar(edad, sexo, imc, hijos, fumador, region):
    fila = pd.DataFrame([{"age": edad, "sex": sexo, "bmi": imc, "children": hijos,
                          "smoker": fumador, "region": region,
                          "fumador_obeso": int(fumador == "yes" and imc >= 30)}])[FEATURES]
    costo = float(modelo_app.predict(fila)[0])
    nota = ("Estimación de apoyo actuarial. No es un diagnóstico ni una decisión de cobertura: "
            "toda recalificación tarifaria relevante requiere revisión del comité técnico.")
    return clp(costo), segmento(costo), nota

app = gr.Interface(
    fn=estimar,
    inputs=[gr.Slider(18, 64, value=35, step=1, label="Edad"),
            gr.Radio([("Hombre", "male"), ("Mujer", "female")], value="female", label="Sexo"),
            gr.Slider(15, 55, value=27, step=0.1, label="IMC"),
            gr.Slider(0, 5, value=1, step=1, label="Número de hijos"),
            gr.Radio([("No", "no"), ("Sí", "yes")], value="no", label="¿Fuma?"),
            gr.Dropdown(["northeast", "northwest", "southeast", "southwest"], value="southeast", label="Región")],
    outputs=[gr.Textbox(label="Costo médico anual esperado (CLP)"),
             gr.Textbox(label="Segmento de riesgo"),
             gr.Textbox(label="Nota de uso")],
    title="Salud Vital S.A. · Estimador de costo médico anual",
    description="Modelo de apoyo para Actuaría, Prevención y Gerencia Comercial.",
    flagging_mode="never",
)
app.launch(share=True)

NameError: name 'joblib' is not defined

## 7. Ficha de implementación técnica (Actividad 4 de la plantilla Excalidraw)
| Punto | Decisión en este notebook | Dónde se ve |
|---|---|---|
| 1. Problema, objetivo y fuga | Regresión sobre `charges`; sin variables derivadas del costo | Sección 2 |
| 2. Partición, trazabilidad, herramientas | 80/20 estratificada, semilla 42, hash del dataset, Python/scikit-learn | Sección 2 |
| 3. Modelo e hiperparámetros | Lineal, Ridge, Random Forest y Gradient Boosting; GridSearch sobre profundidad, tasa y hojas | Sección 2 |
| 4. Métrica y sobreajuste | MAE y R²; brecha train-test y validación cruzada | Sección 3 |
| 5. Monitoreo y automatización | PSI + R² mensual con semáforo y matriz de control | Sección 5 |
| 6. Dashboard y despliegue | Tablero técnico/ejecutivo y app Gradio | Secciones 5 y 6 |